In [38]:
import pandas as pd
import numpy as np

# EDA

In [39]:
df = pd.read_csv("/Users/goeun/Documents/대회/자원 최적화 AI 데이터셋/okm_augumented_2021.csv")
df.head()

,날짜,시간,15분,30분,45분,60분,평균,생산량,기온,풍속,습도,강수량,전기요금(계절),day,d,m,공장인원,인건비
0,20210101,0,62,61,61,61,61,0,-3.2,2.4,71,0.0,109.8,5,1,1,0.0,1.5
1,20210101,1,96,93,116,113,105,0,-4.5,1.5,77,0.0,109.8,5,1,1,0.0,1.5
2,20210101,2,106,96,106,107,104,0,-3.9,2.6,58,0.0,109.8,5,1,1,0.0,1.5
3,20210101,3,92,110,110,109,105,0,-4.1,2.6,56,0.0,109.8,5,1,1,0.0,1.5
4,20210101,4,108,105,106,108,107,0,-4.6,2.6,60,0.0,109.8,5,1,1,0.0,1.5


### 결측치 처리

In [40]:
# 결측 여부 기록
# 나중에 모델이 "원래 값이 없던 행"이라는 사실을 알 수 있게 합니다.
missing_columns = ["풍속", "강수량", "공장인원"]

for column in missing_columns:
    df[f"{column}_결측"] = df[column].isna().astype("int8")


print("처리 전 결측치")
print(df[missing_columns].isna().sum())

처리 전 결측치
풍속       3
강수량      1
공장인원    17
dtype: int64


In [41]:
# 풍속 결측치 처리
# 풍속은 시간에 따라 비교적 연속적으로 변하므로 앞뒤 값 사이를 선형보간합니다.
df["풍속"] = df["풍속"].interpolate(
    method="linear",
    limit_direction="both",
)

In [42]:
# 강수량 결측치 처리
# 같은 날짜 안에서 가장 가까운 유효값을 사용합니다.
# 현재 결측 행은 2021-01-24 00시이며 다음 시간 값이 0이므로 0으로 채워집니다.
df["강수량"] = (
    df.groupby("날짜")["강수량"]
    .transform(lambda series: series.bfill().ffill())
)

# 같은 날짜 전체가 결측인 극단적인 경우에만 0을 사용합니다.
df["강수량"] = df["강수량"].fillna(0)

In [43]:
# 공장인원 결측치 처리
# 현재 17개 결측 행은 생산량이 전부 0이므로 공장인원도 0으로 처리 ? No !!

# 20210828 18시 - 20210829 10시 까지 공장인원 NA
# 다른 6,151개 행에서는 평균전력이 0인 경우가 한 건도 없습니다. 
# 이 데이터에서는 공장이 생산하지 않을 때도 전력을 보통 20 이상 사용합니다.
# 그런데 해당 17시간만 전력이 완전히 0입니다.
# 게다가 같은 시간에 공장인원까지 비어 있습니다.

# 따라서 실제로 전력을 0만큼 사용했다기보다 전력 측정이나 데이터 수집이 끊겼을 가능성이 큽니다.

# 이 17개 행은 삭제하지 말고 유지.
# 전력값 0을 NA로 바꾸고 학습 정답에서 제외.

# 처리 전 20210828 / 18시 / 평균전력 0 / 생산량 0 / 공장인원 NA
# 처리 후 20210828 / 18시 / 평균전력 NA / 생산량 0 / 공장인원 0 / 전력계측공백 1

# 학습할 때는 train_df = df[df["평균"].notna()].copy()

power_columns = ["15분", "30분", "45분", "60분", "평균"]

outage_mask = (
    df[power_columns].eq(0).all(axis=1)
    & df["생산량"].eq(0)
    & df["공장인원"].isna()
)

df["전력계측공백"] = outage_mask.astype(int)

df.loc[outage_mask, power_columns] = pd.NA
df.loc[outage_mask, "공장인원"] = 0


### 이상치 처리

In [44]:
# 2021년 7월 13일: 24건
# 2021년 7월 15일: 24건
# 시간이 0～23이 아니라 70～188로 기록됨

df["시간_수정여부"] = 0

bad_dates = [20210713, 20210715]

for date in bad_dates:
    mask = df["날짜"].eq(date)
    df.loc[mask, "시간"] = range(24)
    df.loc[mask, "시간_수정여부"] = 1

In [45]:
# 삭제하면 안 되는 극단값

# 높은 생산량
# 높은 전력값

# 시간 안에서 큰 전력변화 - 15분 동안 전력이 크게 변하는 행이 있습니다.
# 20210409 7시 (한 시간 안의 범위가 105입니다.)
# 15분 26 → 30분 34 → 45분 63 → 60분 131 
# 이번 프로젝트에서는 오히려 중요한 피크 발생 조건

# 파생변수 생성해서 추후 사용?
# 시간내변동폭 = 최대값 - 최소값 생성

In [46]:
# 의심스럽지만 일단 유지할 값 -> 오류분석 대상

# 생산량이 높은데 전력이 매우 낮은 4건
# 20210205 10시
# 20210205 16시
# 20210219 10시
# 20210226 16시

# 생산량 0인데 높은 전력
# 평균전력 상위 5% 이상인데 생산량이 0인 행이 38건 있습니다.

In [47]:
# 데이터가 증강·복제된 흔적

# 파일명 자체가 augumented이고, 동일한 24시간 전력패턴이 여러 날짜에서 반복됩니다.
# 257일 중 161일은 다른 날짜와 완전히 동일한 일별 평균전력 패턴을 공유

# 이것은 단순 이상치라기보다 데이터 생성 방식의 특징으로 보임 -> 추후 검증

# 새 파일로 저장

In [49]:
# 전처리한 데이터프레임을 새 CSV 파일로 저장
output_path = "okm_augumented_2021_preprocessed.csv"

df.to_csv(
    output_path,
    index=False,          # 불필요한 행 번호를 저장하지 않음
    encoding="utf-8-sig"  # 엑셀에서 한글이 깨지는 것을 방지
)

print(f"저장 완료: {output_path}")

저장 완료: okm_augumented_2021_preprocessed.csv


In [54]:
check_df = pd.read_csv(
    "okm_augumented_2021_preprocessed.csv",
    encoding="utf-8-sig"
)

print("저장된 행 개수:", len(check_df))
print("저장된 열:", check_df.columns.tolist())
print(check_df.head())

print(check_df.isna().sum())

저장된 행 개수: 6168
저장된 열: ['날짜', '시간', '15분', '30분', '45분', '60분', '평균', '생산량', '기온', '풍속', '습도', '강수량', '전기요금(계절)', 'day', 'd', 'm', '공장인원', '인건비', '풍속_결측', '강수량_결측', '공장인원_결측', '전력계측공백', '시간_수정여부']
         날짜  시간    15분    30분    45분    60분     평균  생산량   기온   풍속  ...  day  \
0  20210101   0   62.0   61.0   61.0   61.0   61.0    0 -3.2  2.4  ...    5   
1  20210101   1   96.0   93.0  116.0  113.0  105.0    0 -4.5  1.5  ...    5   
2  20210101   2  106.0   96.0  106.0  107.0  104.0    0 -3.9  2.6  ...    5   
3  20210101   3   92.0  110.0  110.0  109.0  105.0    0 -4.1  2.6  ...    5   
4  20210101   4  108.0  105.0  106.0  108.0  107.0    0 -4.6  2.6  ...    5   

   d  m  공장인원  인건비  풍속_결측  강수량_결측  공장인원_결측  전력계측공백  시간_수정여부  
0  1  1   0.0  1.5      0       0        0       0        0  
1  1  1   0.0  1.5      0       0        0       0        0  
2  1  1   0.0  1.5      0       0        0       0        0  
3  1  1   0.0  1.5      0       0        0       0        0  
4  1  1   0.0  1.5 